In [1]:
#Import the necessary libraries
import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn import datasets

#Probably can use these
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

#Reads
df = pd.read_excel("Bank_Personal_Loan_Modelling.xlsx",
                  sheet_name='Data')


#Checks if there are any missing data in it or duplicate data
duplicate_count = df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)
print("\n")
print(df.isnull().sum())
print("\n")
print((df == "?").sum())

print


df.head()

Number of duplicate rows: 0


ID                    0
Age                   0
Experience            0
Income                0
ZIP Code              0
Family                0
CCAvg                 0
Education             0
Mortgage              0
Personal Loan         0
Securities Account    0
CD Account            0
Online                0
CreditCard            0
dtype: int64


ID                    0
Age                   0
Experience            0
Income                0
ZIP Code              0
Family                0
CCAvg                 0
Education             0
Mortgage              0
Personal Loan         0
Securities Account    0
CD Account            0
Online                0
CreditCard            0
dtype: int64


,ID,Age,Experience,Income,ZIP Code,Family,CCAvg,Education,Mortgage,Personal Loan,Securities Account,CD Account,Online,CreditCard
0,1,25,1,49,91107,4,1.6,1,0,0,1,0,0,0
1,2,45,19,34,90089,3,1.5,1,0,0,1,0,0,0
2,3,39,15,11,94720,1,1.0,1,0,0,0,0,0,0
3,4,35,9,100,94112,1,2.7,2,0,0,0,0,0,0
4,5,35,8,45,91330,4,1.0,2,0,0,0,0,0,1


The section below is the data splitting node.

In [2]:

#This is the section where we are trying to predict whether or not the loans are approved
y = df["Personal Loan"]

#The predictors, while dropping any unnecessary predictors that wouldn't contribute much
x = df.drop(columns=["Personal Loan", "ID", "ZIP Code","Age"])

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.3, random_state=42)

The cell below is where the logistic regression model is used.

In [3]:


# Creates and trains a logistic regression model using the training data.
#the random state makes the model's result reproducible,with the solver as the liblinear

model = LogisticRegression(solver = 'liblinear', random_state = 0).fit(x_train,y_train.values.ravel())

# Calculates the model's accuracy on the training data.
print("The training model's accuracy:", model.score(x_train,y_train))

The training model's accuracy: 0.952


Here is where we are trying to find the most significant variables of the model

In [4]:

#Creates and trains the logistic regression using statsmodel for the logistic regression model
#so that the predictors' coefficients and p-values can be examined

est = sm.Logit(y_train, x_train).fit()

#Creates a table that contains the predictors' coefficients and p-values.
#The coefficient shows the direction of the variable's influence
#while the p-value indicates whether the variable is statistically significant
significance_results = pd.DataFrame({
    "Coefficient": est.params,
    "P-value": est.pvalues
})

#Sorts the variable from the smallest to largest p-value and selects the three most significant predictors
top_three = (
    significance_results
    .sort_values(by="P-value")
    .head(3)
)


#Displays the 3 most significant predictors with smallest p-values with their coefficient
print("Three Most Significant Variables:")
display(top_three)

Optimization terminated successfully.
         Current function value: 0.265117
         Iterations 8
Three Most Significant Variables:


,Coefficient,P-value
CD Account,4.398101,1.981674e-58
Experience,-0.057127,1.515107e-27
Online,-1.335637,2.551999e-25


Now to test the model with test data to see the confusion matrix and the classification report

In [5]:
# Gets the  Probability of Personal Loan = 1
y_probability = model.predict_proba(x_test)[:, 1]


# Set the classification threshold
threshold = [0.40, 0.80]


for i in range(len(threshold)):

    print(f"\nResults for threshold = {threshold[i]}")
    
    # An observation is classified as 1 if
    #its predicted probability is at least greater than or equal to the threshold
    y_pred = (y_probability >= threshold[i]).astype(int)

    #Get the confusion matrix of the true negative and positive as well as the
    #false negative and positive
    print("Confusion Matrix:")
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel().tolist()
    print("True Negative: ",tn) 
    print("False Positive: ",fp)
    print("False Negative: ",fn)
    print("True Positive:", tp)

    #Displays the precision, recall, F1-score, and support for each class.
    #Also displays the total accuracy of the model
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    #This also calculates and displays the accuracy of the prediction from the model by
    # comparing the test data with predictions
    print("The accuracy of the model using test data: ", accuracy_score(y_test,y_pred))



Results for threshold = 0.4
Confusion Matrix:
True Negative:  1319
False Positive:  24
False Negative:  48
True Positive: 109

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97      1343
           1       0.82      0.69      0.75       157

    accuracy                           0.95      1500
   macro avg       0.89      0.84      0.86      1500
weighted avg       0.95      0.95      0.95      1500

The accuracy of the model using test data:  0.952

Results for threshold = 0.8
Confusion Matrix:
True Negative:  1342
False Positive:  1
False Negative:  104
True Positive: 53

Classification Report:
              precision    recall  f1-score   support

           0       0.93      1.00      0.96      1343
           1       0.98      0.34      0.50       157

    accuracy                           0.93      1500
   macro avg       0.95      0.67      0.73      1500
weighted avg       0.93      0.93      0.91      1